# Deepwell pilot: Gemma 4 arms on Colab

Runs each Gemma 4 model in vLLM against the frozen requests in your run folder and writes
`outputs/<arm>.jsonl` straight to Google Drive, so a disconnect loses nothing: re-run the loop cell and it resumes.

**Before you start**
1. On the Mac, follow `eval/README.md` steps 1 to 4: the code is at `MyDrive/deepwell-eval/eval` and the
   prepared run at `MyDrive/deepwell-eval/runs/pilot`.
2. On huggingface.co, accept the Gemma 4 licence, create a read token, and add it in Colab under
   Secrets (key icon) as `HF_TOKEN` with notebook access on.
3. Runtime > Change runtime type: an **80 GB GPU** (H100 or A100 80 GB). On a 40 GB A100 the 31B and 26B A4B
   arms run at 8-bit instead (set automatically, and recorded in the arm name).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture install
!pip install -q vllm
!pip install -q -e /content/drive/MyDrive/deepwell-eval/eval

In [ ]:
import os, subprocess, time, json, urllib.request
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

EVAL = '/content/drive/MyDrive/deepwell-eval/eval'
RUN = '/content/drive/MyDrive/deepwell-eval/runs/pilot'

# arm name -> Hugging Face id. Only gemma-4-31B-it was checked against a model page; the cell below
# confirms the others exist before anything long runs.
MODELS = {
    'gemma-e4b':  'google/gemma-4-E4B-it',
    'gemma-12b':  'google/gemma-4-12B-it',
    'gemma-26b-a4b': 'google/gemma-4-26B-A4B-it',
    'gemma-31b':  'google/gemma-4-31B-it',
}
# Sonnet thinks (effort medium), so the Gemma arms think too. If the smoke test shows reasoning_chars == 0,
# this switch is wrong for Gemma 4's chat template: fix it before the full run.
EXTRA_BODY = {'chat_template_kwargs': {'enable_thinking': True}}
MAX_MODEL_LEN = 24576
GPU_GB = int(subprocess.check_output(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits']).split()[0]) // 1024
BIG = {'gemma-26b-a4b', 'gemma-31b'}
print('GPU memory (GB):', GPU_GB)

In [ ]:
from huggingface_hub import model_info
for arm, mid in MODELS.items():
    try:
        model_info(mid, token=os.environ['HF_TOKEN']); print('ok     ', arm, mid)
    except Exception as e:
        print('MISSING', arm, mid, '->', type(e).__name__, '(fix the id in MODELS before the run)')

In [ ]:
server = None

def start(model_id, quant=None):
    global server
    stop()
    args = ['vllm', 'serve', model_id, '--max-model-len', str(MAX_MODEL_LEN), '--gpu-memory-utilization', '0.92',
            '--reasoning-parser', 'gemma4', '--port', '8000']
    if quant:
        args += ['--quantization', quant]
    log = open(f'/content/vllm-{model_id.split("/")[-1]}.log', 'w')
    server = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):                      # up to 30 min for download and load
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('ready:', model_id); return
        except Exception:
            if server.poll() is not None:
                raise RuntimeError(f'vLLM exited; see {log.name}')
            time.sleep(10)
    raise RuntimeError('vLLM did not become ready in 30 min')

def stop():
    global server
    if server and server.poll() is None:
        server.terminate(); server.wait(60)
    server = None

def gen(arm, model_id, *extra):
    cmd = ['python', '-m', 'deepwell_eval', 'generate', '--run', RUN, '--arm', arm, '--backend', 'openai',
           '--model', model_id, '--extra-body', json.dumps(EXTRA_BODY), '--workers', '8', *extra]
    print(' '.join(cmd[:8]), '...'); subprocess.run(cmd, check=False)

**Smoke test** (about 5 minutes): one request on the smallest model. Check `reasoning_chars` is above 0
(thinking is on) and that `raw` is a JSON menu. Then delete `outputs/smoke-e4b.jsonl` from the run folder.

In [ ]:
start(MODELS['gemma-e4b'])
gen('smoke-e4b', MODELS['gemma-e4b'], '--limit', '1')
row = [json.loads(l) for l in open(f'{RUN}/outputs/smoke-e4b.jsonl')][-1]
print({k: row.get(k) for k in ('stop_reason', 'reasoning_chars', 'seconds', 'usage', 'error')}); print(row.get('raw', '')[:600])

**Full run** (about 2 hours on an 80 GB GPU). Per model: every request with the bundle schema enforced, then the
titles-and-summaries requests without it (`<arm>-raw`, for the format-failure rate). Re-run this cell after a
disconnect: finished requests are skipped.

In [ ]:
for arm, mid in MODELS.items():
    quant = 'fp8' if arm in BIG and GPU_GB < 70 else None
    name = arm + ('-fp8' if quant else '')
    start(mid, quant)
    gen(name, mid)
    gen(name + '-raw', mid, '--unconstrained', '--variants', 'full')
stop()
!ls -la {RUN}/outputs